# 🐍 Python Database Basics
## Module 32 — Complete Learning Book

**Python → SQL → Database → Query → DataFrame**

This module starts with Python's built-in `sqlite3` and introduces the path toward MySQL and PostgreSQL.

### Learning Roadmap

```text
Database Concepts
 ↓
SQL
 ↓
sqlite3
 ↓
Connection + Cursor
 ↓
CRUD
 ↓
Transactions
 ↓
Parameterized SQL
 ↓
JOIN + GROUP BY
 ↓
Keys + Constraints + Indexes
 ↓
Python Database Architecture
 ↓
MySQL / PostgreSQL
 ↓
Real-World Projects
```

# 1. Database Fundamentals

A database stores structured information so applications can persist, query,
update, and analyze data.

### Common relational objects

- Database
- Table
- Row
- Column
- Primary key
- Foreign key
- Index
- Constraint

Example:

```text
customers
orders
products
payments
```

# 2. What Is SQL?

SQL (Structured Query Language) is used with relational databases.

Core operations:

```text
CREATE
INSERT
SELECT
UPDATE
DELETE
```

Analytics commonly uses:

```text
WHERE
GROUP BY
ORDER BY
JOIN
COUNT()
SUM()
AVG()
MIN()
MAX()
```

# 3. What Is SQLite?

SQLite is a lightweight relational database engine.

Python provides the `sqlite3` standard-library module.

```text
Python
 ↓
sqlite3
 ↓
SQL
 ↓
SQLite
 ↓
Rows
```

SQLite is useful for learning, local applications, prototypes, testing, and
embedded database scenarios.

In [1]:
import sqlite3

print("SQLite version:", sqlite3.sqlite_version)
print("sqlite3 loaded successfully")

SQLite version: 3.45.1
sqlite3 loaded successfully


# 4. Connection

Create a database connection:

```python
sqlite3.connect("database.db")
```

For experiments:

```python
sqlite3.connect(":memory:")
```

An in-memory database exists only for the lifetime of the connection.

In [2]:
import sqlite3

connection = sqlite3.connect(":memory:")
print("Connected:", connection is not None)
connection.close()

Connected: True


# 5. Cursor

A cursor executes SQL and retrieves results.

```text
CONNECTION
 ↓
CURSOR
 ↓
EXECUTE SQL
 ↓
FETCH
```

In [3]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute("SELECT 1")
print(cursor.fetchone())

connection.close()

(1,)


# 6. CREATE TABLE

Example:

```sql
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    city TEXT
);
```

`PRIMARY KEY` identifies a row and `NOT NULL` requires a value.

In [4]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute(
    "CREATE TABLE customers ("
    "customer_id INTEGER PRIMARY KEY, "
    "name TEXT NOT NULL, "
    "city TEXT)"
)

connection.commit()
print("Table created")

connection.close()

Table created


# 7. INSERT

Use parameterized values:

```python
cursor.execute(
    "INSERT INTO customers (name, city) VALUES (?, ?)",
    ("Kaif", "Hyderabad")
)
```

The `?` placeholders are SQLite parameter markers.

In [5]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute(
    "CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT)"
)

cursor.execute(
    "INSERT INTO customers (name, city) VALUES (?, ?)",
    ("Kaif", "Hyderabad"),
)

connection.commit()

cursor.execute("SELECT * FROM customers")
print(cursor.fetchall())

connection.close()

[(1, 'Kaif', 'Hyderabad')]


# 8. `executemany()`

Use `executemany()` for repeated execution of one parameterized statement.

In [6]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute(
    "CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, price REAL)"
)

products = [
    ("Laptop", 65000.0),
    ("Mouse", 1200.0),
    ("Keyboard", 2500.0),
]

cursor.executemany(
    "INSERT INTO products (name, price) VALUES (?, ?)",
    products,
)

connection.commit()

cursor.execute("SELECT * FROM products ORDER BY id")

for row in cursor.fetchall():
    print(row)

connection.close()

(1, 'Laptop', 65000.0)
(2, 'Mouse', 1200.0)
(3, 'Keyboard', 2500.0)


# 9. SELECT

Retrieve records with `SELECT`.

Prefer selecting only required columns when practical.

In [7]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute(
    "CREATE TABLE products (name TEXT, price REAL)"
)

cursor.executemany(
    "INSERT INTO products VALUES (?, ?)",
    [
        ("Laptop", 65000.0),
        ("Mouse", 1200.0),
        ("Keyboard", 2500.0),
    ],
)

connection.commit()

cursor.execute("SELECT name, price FROM products ORDER BY price DESC")

for row in cursor.fetchall():
    print(row)

connection.close()

('Laptop', 65000.0)
('Keyboard', 2500.0)
('Mouse', 1200.0)


# 10. Fetch Methods

### `fetchone()`
Returns one remaining row.

### `fetchmany(size)`
Returns up to `size` remaining rows.

### `fetchall()`
Returns all remaining rows.

Choose based on the amount of data you need.

In [8]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute("CREATE TABLE numbers (value INTEGER)")
cursor.executemany("INSERT INTO numbers VALUES (?)", [(10,), (20,), (30,), (40,)])
connection.commit()

cursor.execute("SELECT value FROM numbers ORDER BY value")

print("one:", cursor.fetchone())
print("many:", cursor.fetchmany(2))
print("all:", cursor.fetchall())

connection.close()

one: (10,)
many: [(20,), (30,)]
all: [(40,)]


# 11. WHERE

Filter records:

```sql
SELECT name, price
FROM products
WHERE price > ?;
```

Pass values separately as parameters.

In [9]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute("CREATE TABLE products (name TEXT, price REAL)")
cursor.executemany(
    "INSERT INTO products VALUES (?, ?)",
    [("Laptop", 65000.0), ("Mouse", 1200.0), ("Keyboard", 2500.0)],
)
connection.commit()

cursor.execute(
    "SELECT name, price FROM products WHERE price > ?",
    (2000.0,),
)

print(cursor.fetchall())
connection.close()

[('Laptop', 65000.0), ('Keyboard', 2500.0)]


# 12. UPDATE

Modify existing rows:

```sql
UPDATE products
SET price = ?
WHERE name = ?;
```

Always check the condition before executing a production update.

In [10]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute("CREATE TABLE products (name TEXT, price REAL)")
cursor.execute("INSERT INTO products VALUES (?, ?)", ("Mouse", 1200.0))

cursor.execute(
    "UPDATE products SET price = ? WHERE name = ?",
    (1500.0, "Mouse"),
)

connection.commit()

print(cursor.execute("SELECT * FROM products").fetchall())
connection.close()

[('Mouse', 1500.0)]


# 13. DELETE

Delete records:

```sql
DELETE FROM products
WHERE name = ?;
```

Be especially careful with DELETE statements that lack an intended filter.

In [11]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute("CREATE TABLE products (name TEXT, price REAL)")
cursor.executemany(
    "INSERT INTO products VALUES (?, ?)",
    [("Laptop", 65000.0), ("Mouse", 1200.0)],
)

cursor.execute("DELETE FROM products WHERE name = ?", ("Mouse",))
connection.commit()

print(cursor.execute("SELECT * FROM products").fetchall())
connection.close()

[('Laptop', 65000.0)]


# 14. Transactions — `commit()` and `rollback()`

```text
WORK
 ↓
SUCCESS → COMMIT
 ↓
FAILURE → ROLLBACK
```

- `commit()` saves the transaction.
- `rollback()` rolls back uncommitted changes.

In [12]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute("CREATE TABLE accounts (id INTEGER PRIMARY KEY, balance REAL)")
cursor.execute("INSERT INTO accounts VALUES (?, ?)", (1, 10000.0))
connection.commit()

try:
    cursor.execute(
        "UPDATE accounts SET balance = balance - ? WHERE id = ?",
        (2500.0, 1),
    )
    connection.commit()
except sqlite3.Error:
    connection.rollback()
    raise

print(cursor.execute("SELECT * FROM accounts").fetchall())
connection.close()

[(1, 7500.0)]


# 15. Context Manager

A connection can be used with:

```python
with sqlite3.connect("app.db") as connection:
    ...
```

This gives a clean transaction-management pattern for many operations.

In [13]:
import sqlite3

with sqlite3.connect(":memory:") as connection:
    cursor = connection.cursor()
    cursor.execute("CREATE TABLE sales (amount REAL)")
    cursor.executemany("INSERT INTO sales VALUES (?)", [(1000.0,), (2500.0,)])
    cursor.execute("SELECT SUM(amount) FROM sales")
    print("Total:", cursor.fetchone()[0])

Total: 3500.0


# 16. Parameterized Queries and SQL Injection

### Avoid

```python
sql = f"SELECT * FROM customers WHERE name = '{name}'"
```

### Prefer

```python
cursor.execute(
    "SELECT * FROM customers WHERE name = ?",
    (name,)
)
```

Parameterized SQL separates values from SQL syntax and is an important
security practice.

In [14]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute("CREATE TABLE customers (name TEXT, city TEXT)")
cursor.execute(
    "INSERT INTO customers VALUES (?, ?)",
    ("Kaif", "Hyderabad"),
)
connection.commit()

name = "Kaif"

cursor.execute(
    "SELECT name, city FROM customers WHERE name = ?",
    (name,),
)

print(cursor.fetchone())
connection.close()

('Kaif', 'Hyderabad')


# 17. `sqlite3.Row`

By default, rows are tuple-like.

Configure:

```python
connection.row_factory = sqlite3.Row
```

Then access values by column name.

In [15]:
import sqlite3

connection = sqlite3.connect(":memory:")
connection.row_factory = sqlite3.Row
cursor = connection.cursor()

cursor.execute("CREATE TABLE customers (id INTEGER, name TEXT, city TEXT)")
cursor.execute("INSERT INTO customers VALUES (?, ?, ?)", (101, "Kaif", "Hyderabad"))
connection.commit()

row = cursor.execute("SELECT * FROM customers").fetchone()

print(row["id"])
print(row["name"])
print(row["city"])

connection.close()

101
Kaif
Hyderabad


# 18. Aggregate Functions

Important SQL aggregates:

```text
COUNT()
SUM()
AVG()
MIN()
MAX()
```

These are highly useful for analytics and reporting.

In [16]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute("CREATE TABLE sales (amount REAL)")
cursor.executemany(
    "INSERT INTO sales VALUES (?)",
    [(1200.0,), (2500.0,), (1800.0,), (3200.0,)],
)
connection.commit()

cursor.execute(
    "SELECT COUNT(*), SUM(amount), AVG(amount), MIN(amount), MAX(amount) FROM sales"
)

print(cursor.fetchone())
connection.close()

(4, 8700.0, 2175.0, 1200.0, 3200.0)


# 19. GROUP BY

`GROUP BY` calculates aggregates per group.

```sql
SELECT city, SUM(amount)
FROM sales
GROUP BY city;
```

In [17]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute("CREATE TABLE sales (city TEXT, amount REAL)")
cursor.executemany(
    "INSERT INTO sales VALUES (?, ?)",
    [
        ("Hyderabad", 1200.0),
        ("Hyderabad", 2500.0),
        ("Mumbai", 1800.0),
        ("Mumbai", 3200.0),
    ],
)
connection.commit()

cursor.execute(
    "SELECT city, SUM(amount) AS total_sales "
    "FROM sales GROUP BY city ORDER BY total_sales DESC"
)

print(cursor.fetchall())
connection.close()

[('Mumbai', 5000.0), ('Hyderabad', 3700.0)]


# 20. JOIN

A JOIN combines related tables.

```text
customers
    ↓ customer_id
orders
```

This is one of the most important relational database concepts.

In [18]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute("CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT)")
cursor.execute(
    "CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, amount REAL)"
)

cursor.executemany("INSERT INTO customers VALUES (?, ?)", [(1, "Kaif"), (2, "Sara")])
cursor.executemany(
    "INSERT INTO orders (customer_id, amount) VALUES (?, ?)",
    [(1, 2500.0), (1, 1800.0), (2, 3200.0)],
)
connection.commit()

cursor.execute(
    "SELECT customers.name, orders.amount "
    "FROM customers JOIN orders "
    "ON customers.id = orders.customer_id "
    "ORDER BY customers.name"
)

print(cursor.fetchall())
connection.close()

[('Kaif', 2500.0), ('Kaif', 1800.0), ('Sara', 3200.0)]


# 21. Primary Keys and Foreign Keys

### Primary key

Uniquely identifies a row.

### Foreign key

References a key in another table.

```text
customers.id
     ↑
     │
orders.customer_id
```

SQLite foreign-key enforcement can be enabled with:

```sql
PRAGMA foreign_keys = ON;
```

In [19]:
import sqlite3

connection = sqlite3.connect(":memory:")
connection.execute("PRAGMA foreign_keys = ON")

cursor = connection.cursor()

cursor.execute(
    "CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT NOT NULL)"
)

cursor.execute(
    "CREATE TABLE orders ("
    "id INTEGER PRIMARY KEY, "
    "customer_id INTEGER NOT NULL, "
    "amount REAL NOT NULL, "
    "FOREIGN KEY(customer_id) REFERENCES customers(id))"
)

print("Schema created")
connection.close()

Schema created


# 22. Constraints

Important constraints:

```text
PRIMARY KEY
FOREIGN KEY
NOT NULL
UNIQUE
CHECK
```

Constraints help the database protect data integrity.

In [20]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute(
    "CREATE TABLE products ("
    "id INTEGER PRIMARY KEY, "
    "sku TEXT UNIQUE NOT NULL, "
    "price REAL CHECK(price >= 0))"
)

cursor.execute(
    "INSERT INTO products (sku, price) VALUES (?, ?)",
    ("SKU-001", 1200.0),
)

connection.commit()

print(cursor.execute("SELECT * FROM products").fetchall())
connection.close()

[(1, 'SKU-001', 1200.0)]


# 23. Indexes

Indexes can improve performance for suitable queries.

Example:

```sql
CREATE INDEX idx_customer_email
ON customers(email);
```

Indexes consume storage and add write-maintenance cost, so create them based
on actual query requirements.

In [21]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute("CREATE TABLE customers (id INTEGER PRIMARY KEY, email TEXT)")
cursor.execute("CREATE INDEX idx_customer_email ON customers(email)")

print("Index created")
connection.close()

Index created


# 24. Database Errors

Useful SQLite exception types include:

```text
sqlite3.Error
sqlite3.IntegrityError
sqlite3.OperationalError
sqlite3.ProgrammingError
```

Handle exceptions when you can recover, report, or clean up meaningfully.

In [22]:
import sqlite3

connection = sqlite3.connect(":memory:")

try:
    connection.execute("SELECT * FROM missing_table")
except sqlite3.OperationalError as error:
    print("Database error:", error)
finally:
    connection.close()

Database error: no such table: missing_table


# 25. Database → Python → DataFrame

For data analytics, a common architecture is:

```text
PYTHON
 ↓
SQL QUERY
 ↓
DATABASE
 ↓
RESULT SET
 ↓
PYTHON RECORDS
 ↓
DATAFRAME
 ↓
ANALYSIS
 ↓
VISUALIZATION / ML
```

A DataFrame is an optional downstream analytics layer.

In [23]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute("CREATE TABLE sales (region TEXT, amount REAL)")
cursor.executemany(
    "INSERT INTO sales VALUES (?, ?)",
    [("South", 65000.0), ("South", 18000.0), ("West", 42000.0)],
)
connection.commit()

cursor.execute(
    "SELECT region, SUM(amount) AS total_sales "
    "FROM sales GROUP BY region ORDER BY total_sales DESC"
)

columns = [item[0] for item in cursor.description]
records = [dict(zip(columns, row)) for row in cursor.fetchall()]

print(records)
connection.close()

# Optional:
# Convert `records` into a DataFrame in an analytics project.

[{'region': 'South', 'total_sales': 83000.0}, {'region': 'West', 'total_sales': 42000.0}]


# 26. MySQL — Next Step

MySQL is a server-based relational database.

Typical architecture:

```text
Python
 ↓
MySQL Driver
 ↓
MySQL Server
 ↓
SQL
 ↓
Results
```

A Python MySQL project needs a compatible driver/client library. The exact
driver and connection configuration depend on the project.

# 27. PostgreSQL — Next Step

PostgreSQL is an open-source relational database system.

Typical architecture:

```text
Python
 ↓
PostgreSQL Driver
 ↓
PostgreSQL Server
 ↓
SQL
 ↓
Results
```

PostgreSQL provides a broad range of advanced database features.

# 28. SQLite vs MySQL vs PostgreSQL

| Feature | SQLite | MySQL | PostgreSQL |
|---|---|---|---|
| Embedded | Yes | No | No |
| Separate server | No | Yes | Yes |
| Learning SQL | Excellent | Excellent | Excellent |
| Local applications | Excellent | Possible | Possible |
| Server workloads | More limited | Strong | Strong |
| Python driver | Standard `sqlite3` | External driver | External driver |

Choose based on requirements rather than popularity alone.

# 29. Professional Database Architecture

For larger applications:

```text
APPLICATION
     ↓
SERVICE / BUSINESS LOGIC
     ↓
REPOSITORY / DATA ACCESS
     ↓
DATABASE DRIVER
     ↓
DATABASE
```

This separation keeps database concerns organized.

# 30. Common Mistakes

| Problem | Why | Better Approach |
|---|---|---|
| Forgetting `commit()` | Changes may not persist | Commit intentionally |
| SQL concatenation | Injection risk | Parameterized SQL |
| Missing `WHERE` | Too many rows can change | Verify filters |
| Huge result sets | Memory/performance cost | Select only needed data |
| Ignoring transactions | Partial updates | Use transaction boundaries |
| Hard-coded secrets | Credential exposure | Environment/secrets management |
| Too many indexes | Write/storage cost | Index actual query patterns |

# 31. Best Practices

- Use parameterized SQL.
- Keep transactions deliberate.
- Use constraints for important data rules.
- Validate external input.
- Select only needed columns.
- Index based on query patterns.
- Keep SQL readable.
- Separate database access from business logic.
- Test database operations.
- Manage connections carefully.
- Never put passwords or API keys in source code.
- Choose SQLite, MySQL, or PostgreSQL according to requirements.

# 🚀 Project 1 — Customer CRUD Database

## Problem

Create, read, update, and delete customer records.

## Architecture

```text
INPUT
 ↓
VALIDATION
 ↓
PYTHON
 ↓
SQL
 ↓
DATABASE
 ↓
QUERY
 ↓
RESULT
 ↓
REPORT / ANALYTICS
```

## Implementation

In [24]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute(
    "CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT NOT NULL, city TEXT NOT NULL)"
)

cursor.executemany(
    "INSERT INTO customers (name, city) VALUES (?, ?)",
    [("Kaif", "Hyderabad"), ("Sara", "Mumbai"), ("Ali", "Delhi")],
)
connection.commit()

print("Initial:", cursor.execute("SELECT * FROM customers").fetchall())

cursor.execute(
    "UPDATE customers SET city = ? WHERE name = ?",
    ("Pune", "Ali"),
)
cursor.execute("DELETE FROM customers WHERE name = ?", ("Sara",))
connection.commit()

print("Final:", cursor.execute("SELECT * FROM customers ORDER BY id").fetchall())
connection.close()

Initial: [(1, 'Kaif', 'Hyderabad'), (2, 'Sara', 'Mumbai'), (3, 'Ali', 'Delhi')]
Final: [(1, 'Kaif', 'Hyderabad'), (3, 'Ali', 'Pune')]


# 🚀 Project 2 — Sales KPI Database

## Problem

Calculate core business KPIs with SQL.

## Architecture

```text
INPUT
 ↓
VALIDATION
 ↓
PYTHON
 ↓
SQL
 ↓
DATABASE
 ↓
QUERY
 ↓
RESULT
 ↓
REPORT / ANALYTICS
```

## Implementation

In [25]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute("CREATE TABLE sales (customer TEXT, amount REAL)")
cursor.executemany(
    "INSERT INTO sales VALUES (?, ?)",
    [
        ("Kaif", 65000.0),
        ("Sara", 18000.0),
        ("Ali", 42000.0),
        ("Maya", 25000.0),
    ],
)
connection.commit()

cursor.execute(
    "SELECT COUNT(*), SUM(amount), AVG(amount), MIN(amount), MAX(amount) FROM sales"
)

count, total, average, minimum, maximum = cursor.fetchone()

print("Transactions:", count)
print("Total:", total)
print("Average:", average)
print("Minimum:", minimum)
print("Maximum:", maximum)

connection.close()

Transactions: 4
Total: 150000.0
Average: 37500.0
Minimum: 18000.0
Maximum: 65000.0


# 🚀 Project 3 — Inventory Management

## Problem

Reduce stock safely after an inventory sale.

## Architecture

```text
INPUT
 ↓
VALIDATION
 ↓
PYTHON
 ↓
SQL
 ↓
DATABASE
 ↓
QUERY
 ↓
RESULT
 ↓
REPORT / ANALYTICS
```

## Implementation

In [26]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute(
    "CREATE TABLE inventory ("
    "id INTEGER PRIMARY KEY, "
    "product TEXT UNIQUE NOT NULL, "
    "stock INTEGER NOT NULL CHECK(stock >= 0), "
    "price REAL NOT NULL CHECK(price >= 0))"
)

cursor.executemany(
    "INSERT INTO inventory (product, stock, price) VALUES (?, ?, ?)",
    [
        ("Laptop", 10, 65000.0),
        ("Mouse", 25, 1200.0),
        ("Keyboard", 15, 2500.0),
    ],
)
connection.commit()

product = "Laptop"
quantity = 2

cursor.execute("SELECT stock FROM inventory WHERE product = ?", (product,))
row = cursor.fetchone()

if row is None:
    raise ValueError("Product not found")

if row[0] < quantity:
    raise ValueError("Insufficient stock")

cursor.execute(
    "UPDATE inventory SET stock = stock - ? WHERE product = ?",
    (quantity, product),
)
connection.commit()

print(cursor.execute("SELECT * FROM inventory ORDER BY product").fetchall())
connection.close()

[(3, 'Keyboard', 15, 2500.0), (1, 'Laptop', 8, 65000.0), (2, 'Mouse', 25, 1200.0)]


# 🚀 Project 4 — Customer Orders JOIN

## Problem

Calculate customer spending by joining customer and order tables.

## Architecture

```text
INPUT
 ↓
VALIDATION
 ↓
PYTHON
 ↓
SQL
 ↓
DATABASE
 ↓
QUERY
 ↓
RESULT
 ↓
REPORT / ANALYTICS
```

## Implementation

In [27]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute("CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT NOT NULL)")
cursor.execute(
    "CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER NOT NULL, amount REAL NOT NULL)"
)

cursor.executemany("INSERT INTO customers VALUES (?, ?)", [(1, "Kaif"), (2, "Sara"), (3, "Ali")])
cursor.executemany(
    "INSERT INTO orders (customer_id, amount) VALUES (?, ?)",
    [(1, 65000.0), (1, 18000.0), (2, 25000.0), (3, 42000.0)],
)
connection.commit()

cursor.execute(
    "SELECT customers.name, SUM(orders.amount) AS total_spend "
    "FROM customers JOIN orders ON customers.id = orders.customer_id "
    "GROUP BY customers.id, customers.name "
    "ORDER BY total_spend DESC"
)

for row in cursor.fetchall():
    print(row)

connection.close()

('Kaif', 83000.0)
('Ali', 42000.0)
('Sara', 25000.0)


# 🚀 Project 5 — Transaction-Safe Transfer

## Problem

Transfer money between accounts as one controlled transaction.

## Architecture

```text
INPUT
 ↓
VALIDATION
 ↓
PYTHON
 ↓
SQL
 ↓
DATABASE
 ↓
QUERY
 ↓
RESULT
 ↓
REPORT / ANALYTICS
```

## Implementation

In [28]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute(
    "CREATE TABLE accounts (id INTEGER PRIMARY KEY, owner TEXT, balance REAL CHECK(balance >= 0))"
)

cursor.executemany(
    "INSERT INTO accounts VALUES (?, ?, ?)",
    [(1, "Kaif", 10000.0), (2, "Sara", 5000.0)],
)
connection.commit()

amount = 2500.0

try:
    cursor.execute("SELECT balance FROM accounts WHERE id = ?", (1,))
    sender = cursor.fetchone()

    if sender is None:
        raise ValueError("Sender not found")
    if sender[0] < amount:
        raise ValueError("Insufficient balance")

    cursor.execute(
        "UPDATE accounts SET balance = balance - ? WHERE id = ?",
        (amount, 1),
    )
    cursor.execute(
        "UPDATE accounts SET balance = balance + ? WHERE id = ?",
        (amount, 2),
    )

    connection.commit()

except (sqlite3.Error, ValueError):
    connection.rollback()
    raise

print(cursor.execute("SELECT * FROM accounts ORDER BY id").fetchall())
connection.close()

[(1, 'Kaif', 7500.0), (2, 'Sara', 7500.0)]


# 🚀 Project 6 — Parameterized Customer Search

## Problem

Search customer records safely.

## Architecture

```text
INPUT
 ↓
VALIDATION
 ↓
PYTHON
 ↓
SQL
 ↓
DATABASE
 ↓
QUERY
 ↓
RESULT
 ↓
REPORT / ANALYTICS
```

## Implementation

In [29]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute("CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT)")
cursor.executemany(
    "INSERT INTO customers (name, city) VALUES (?, ?)",
    [
        ("Kaif", "Hyderabad"),
        ("Sara", "Mumbai"),
        ("Ali", "Hyderabad"),
        ("Maya", "Delhi"),
    ],
)
connection.commit()

city = "Hyderabad"

cursor.execute(
    "SELECT id, name, city FROM customers WHERE city = ? ORDER BY name",
    (city,),
)

for row in cursor.fetchall():
    print(row)

connection.close()

(3, 'Ali', 'Hyderabad')
(1, 'Kaif', 'Hyderabad')


# 🚀 Project 7 — SQLite Analytics Report

## Problem

Generate a region-level sales report.

## Architecture

```text
INPUT
 ↓
VALIDATION
 ↓
PYTHON
 ↓
SQL
 ↓
DATABASE
 ↓
QUERY
 ↓
RESULT
 ↓
REPORT / ANALYTICS
```

## Implementation

In [30]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute(
    "CREATE TABLE sales (salesperson TEXT, region TEXT, amount REAL)"
)

cursor.executemany(
    "INSERT INTO sales VALUES (?, ?, ?)",
    [
        ("Kaif", "South", 65000.0),
        ("Sara", "West", 18000.0),
        ("Ali", "South", 42000.0),
        ("Maya", "North", 25000.0),
        ("Kaif", "South", 32000.0),
    ],
)
connection.commit()

cursor.execute(
    "SELECT COUNT(*), SUM(amount), AVG(amount), MAX(amount) FROM sales"
)

print("KPIs:", cursor.fetchone())

cursor.execute(
    "SELECT region, SUM(amount) AS total_sales "
    "FROM sales GROUP BY region ORDER BY total_sales DESC"
)

print("Regions:")
for row in cursor.fetchall():
    print(row)

connection.close()

KPIs: (5, 182000.0, 36400.0, 65000.0)
Regions:
('South', 139000.0)
('North', 25000.0)
('West', 18000.0)


# 🚀 Project 8 — SQLite Repository Pattern

## Problem

Separate data-access logic from application code.

## Architecture

```text
INPUT
 ↓
VALIDATION
 ↓
PYTHON
 ↓
SQL
 ↓
DATABASE
 ↓
QUERY
 ↓
RESULT
 ↓
REPORT / ANALYTICS
```

## Implementation

In [31]:
import sqlite3

class CustomerRepository:
    def __init__(self, connection):
        self.connection = connection

    def create_table(self):
        self.connection.execute(
            "CREATE TABLE IF NOT EXISTS customers ("
            "id INTEGER PRIMARY KEY, name TEXT NOT NULL, city TEXT NOT NULL)"
        )
        self.connection.commit()

    def add(self, name, city):
        self.connection.execute(
            "INSERT INTO customers (name, city) VALUES (?, ?)",
            (name, city),
        )
        self.connection.commit()

    def list_all(self):
        return self.connection.execute(
            "SELECT id, name, city FROM customers ORDER BY id"
        ).fetchall()


connection = sqlite3.connect(":memory:")
repo = CustomerRepository(connection)

repo.create_table()
repo.add("Kaif", "Hyderabad")
repo.add("Sara", "Mumbai")

for row in repo.list_all():
    print(row)

connection.close()

(1, 'Kaif', 'Hyderabad')
(2, 'Sara', 'Mumbai')


# 🚀 Project 9 — Database-to-Analytics Pipeline

## Problem

Turn SQL results into Python records ready for downstream analytics.

## Architecture

```text
INPUT
 ↓
VALIDATION
 ↓
PYTHON
 ↓
SQL
 ↓
DATABASE
 ↓
QUERY
 ↓
RESULT
 ↓
REPORT / ANALYTICS
```

## Implementation

In [32]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute("CREATE TABLE sales (region TEXT, amount REAL)")
cursor.executemany(
    "INSERT INTO sales VALUES (?, ?)",
    [
        ("South", 65000.0),
        ("South", 18000.0),
        ("West", 42000.0),
        ("North", 25000.0),
    ],
)
connection.commit()

cursor.execute(
    "SELECT region, SUM(amount) AS total_sales "
    "FROM sales GROUP BY region ORDER BY total_sales DESC"
)

columns = [item[0] for item in cursor.description]
records = [dict(zip(columns, row)) for row in cursor.fetchall()]

print(records)
connection.close()

# In an analytics project, `records` can be converted to a DataFrame.

[{'region': 'South', 'total_sales': 83000.0}, {'region': 'West', 'total_sales': 42000.0}, {'region': 'North', 'total_sales': 25000.0}]


# 🚀 Project 10 — Production-Style SQLite Sales App

## Problem

Combine validation, transactions, repository logic, schema constraints, and reporting.

## Architecture

```text
INPUT
 ↓
VALIDATION
 ↓
PYTHON
 ↓
SQL
 ↓
DATABASE
 ↓
QUERY
 ↓
RESULT
 ↓
REPORT / ANALYTICS
```

## Implementation

In [33]:
import sqlite3
from typing import TypedDict

class Sale(TypedDict):
    customer: str
    region: str
    amount: float

class SalesRepository:
    def __init__(self, connection: sqlite3.Connection):
        self.connection = connection

    def create_schema(self) -> None:
        self.connection.execute(
            "CREATE TABLE IF NOT EXISTS sales ("
            "id INTEGER PRIMARY KEY, "
            "customer TEXT NOT NULL, "
            "region TEXT NOT NULL, "
            "amount REAL NOT NULL CHECK(amount >= 0))"
        )
        self.connection.commit()

    def add_sale(self, sale: Sale) -> None:
        customer = sale["customer"].strip()
        region = sale["region"].strip()

        if not customer:
            raise ValueError("Customer is required")
        if not region:
            raise ValueError("Region is required")
        if sale["amount"] < 0:
            raise ValueError("Amount cannot be negative")

        self.connection.execute(
            "INSERT INTO sales (customer, region, amount) VALUES (?, ?, ?)",
            (customer, region, sale["amount"]),
        )

    def report(self) -> dict[str, float | int]:
        row = self.connection.execute(
            "SELECT COUNT(*), COALESCE(SUM(amount), 0), COALESCE(AVG(amount), 0) FROM sales"
        ).fetchone()

        return {"count": row[0], "total": row[1], "average": row[2]}

def main() -> None:
    connection = sqlite3.connect(":memory:")

    try:
        repo = SalesRepository(connection)
        repo.create_schema()

        sales: list[Sale] = [
            {"customer": "Kaif", "region": "South", "amount": 65000.0},
            {"customer": "Sara", "region": "West", "amount": 18000.0},
            {"customer": "Ali", "region": "South", "amount": 42000.0},
        ]

        for sale in sales:
            repo.add_sale(sale)

        connection.commit()
        print(repo.report())

    except (sqlite3.Error, ValueError):
        connection.rollback()
        raise
    finally:
        connection.close()

if __name__ == "__main__":
    main()

{'count': 3, 'total': 125000.0, 'average': 41666.666666666664}


# 🧪 Practice — Beginner → Advanced

## Beginner

1. What is a database?
2. What is SQL?
3. What is SQLite?
4. What is `sqlite3`?
5. What is a connection?
6. What is a cursor?
7. Create a table.
8. Insert records.
9. Run SELECT.
10. Use `fetchone()` and `fetchall()`.

## Intermediate

11. Use WHERE.
12. Use UPDATE.
13. Use DELETE.
14. Explain CRUD.
15. Explain commit and rollback.
16. Use parameterized SQL.
17. Explain SQL injection.
18. Use GROUP BY.
19. Use aggregate functions.
20. Perform a JOIN.
21. Use primary and foreign keys.
22. Add constraints.
23. Create an index.

## Advanced

24. Design a relational schema.
25. Explain transaction boundaries.
26. Build a repository/data-access layer.
27. Build a database-to-analytics pipeline.
28. Compare SQLite, MySQL, and PostgreSQL.
29. Decide when SQLite is no longer appropriate.
30. Design a production-style Python database layer.

# 🎤 Interview Questions

1. What is SQLite?
2. What is `sqlite3`?
3. What is a cursor?
4. What does `execute()` do?
5. Difference between `fetchone()` and `fetchall()`?
6. What is a transaction?
7. What does `commit()` do?
8. What does `rollback()` do?
9. Why use parameterized queries?
10. What is SQL injection?
11. What is a primary key?
12. What is a foreign key?
13. What is an index?
14. Why can too many indexes be harmful?
15. What is a JOIN?
16. What is GROUP BY?
17. When is SQLite appropriate?
18. When would PostgreSQL be preferred?
19. How would you handle a failed transaction?
20. How would you move SQL results into a DataFrame?
21. How would you structure database access in a large Python project?

# 📚 A–Z Quick Reference

| Letter | Concept |
|---|---|
| A | Aggregation |
| B | Business data |
| C | Connection / Cursor / Commit |
| D | Database |
| E | Execute |
| F | Fetch |
| G | GROUP BY |
| H | JOIN |
| I | Index |
| J | JOIN |
| K | Keys |
| L | SQLite |
| M | MySQL |
| N | NULL |
| O | ORDER BY |
| P | PostgreSQL / Parameterized SQL |
| Q | Query |
| R | Rollback / Row |
| S | SELECT / SQLite |
| T | Transaction / Table |
| U | UPDATE |
| V | Validation |
| W | WHERE |
| X | SQL execution |
| Y | Python → SQL pipeline |
| Z | Safe input handling |

### Most Used Pattern

```python
import sqlite3

with sqlite3.connect("app.db") as connection:
    rows = connection.execute(
        "SELECT id, name FROM customers WHERE city = ?",
        ("Hyderabad",),
    ).fetchall()

for row in rows:
    print(row)
```

### Analytics Pattern

```text
PYTHON
 ↓
SQL QUERY
 ↓
DATABASE
 ↓
RESULT SET
 ↓
PYTHON RECORDS
 ↓
DATAFRAME
 ↓
ANALYSIS
 ↓
VISUALIZATION / ML
```

# 🎯 Final Master Roadmap

```text
DATABASE
 ↓
SQL
 ↓
sqlite3
 ↓
CONNECTION
 ↓
CURSOR
 ↓
CREATE TABLE
 ↓
INSERT
 ↓
SELECT
 ↓
WHERE
 ↓
UPDATE
 ↓
DELETE
 ↓
FETCH
 ↓
TRANSACTIONS
 ↓
COMMIT / ROLLBACK
 ↓
PARAMETERIZED SQL
 ↓
JOIN
 ↓
GROUP BY
 ↓
KEYS / CONSTRAINTS
 ↓
INDEXES
 ↓
DATABASE ARCHITECTURE
 ↓
PYTHON → SQL → DATABASE
 ↓
DATABASE → PYTHON
 ↓
DATAFRAME
 ↓
MYSQL
 ↓
POSTGRESQL
 ↓
PROJECTS
 ↓
PROFESSIONAL DATABASE DEVELOPMENT
```

# 🎯 The END → Next Journey Begins

**Thank you for following this Jupyter Notebook.**

Keep learning, keep practicing, and keep building real-world projects.

> **Learn → Practice → Analyze → Build → Improve → Grow**

See you in the next notebook. 🚀

### Until then, keep coding and keep learning! 💻🐍

**— S Mohammed Kaif**

---

<div align="center">

## 👨‍💻 S Mohammed Kaif

**Data Science • Data Analytics • Machine Learning • AI • Python**

GitHub: https://github.com/Shaik-Mohammed-Kaif

LinkedIn: https://www.linkedin.com/in/s-mohammed-kaif-2a500a341/

**© 2026 S Mohammed Kaif**

</div>